In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# A5 four-way boundary experiment — paper-aligned head

This notebook tests a non-Abelian, higher-dimensional boundary task using the alternating group \(A_5\), which has 60 elements and 3,600 ordered multiplication pairs.

The four conditions independently cross:

- residual geometry: `layernorm` vs `spherical`
- output head: `raw` vs normalized-unembedding `bounded`

The bounded head matches the paper, Addition, and S5 implementations: it normalizes every unembedding row and applies a fixed multiplier `tau=10`, without separately normalizing the final hidden state. For spherical residuals the final hidden state is already unit norm, so this is exactly a fixed-temperature cosine classifier. For LayerNorm, hidden-state norm remains part of the model.

Legacy `a5_sphere_cosine_wd1` files are reused because that spherical computation is identical. Legacy `a5_layernorm_cosine_wd1` files are deliberately excluded; the paper-aligned LayerNorm control uses the new `a5_layernorm_bounded_wd1` name.

All default runs use 40% training coverage, full-batch AdamW, `lr=1e-3`, `weight_decay=1`, `beta2=0.98`, 250,000 epochs, and five matched seeds. Atomic Google Drive checkpoints support clean resume after Colab interruptions.


In [ ]:
import os
import json
import math
import random
import time
import itertools
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

SAVE_ROOT = "/content/drive/MyDrive/grokking_rebuttal_A5_fourway_v1"
os.makedirs(SAVE_ROOT, exist_ok=True)

DEGREE = 5
GROUP_NAME = "A5"
GROUP_SIZE = math.factorial(DEGREE) // 2  # 60
OP_TOKEN = GROUP_SIZE

FRAC_TRAIN = 0.40
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
N_LAYERS = 1

LR = 1e-3
WEIGHT_DECAY = 1.0
ADAM_BETAS = (0.9, 0.98)
MAX_EPOCHS = 250_000
SEEDS = [1, 2, 3, 4, 5]

LOG_EVERY = 200
CHECKPOINT_EVERY = 2_500
POST_99_GRACE = 1_000
STABLE_99_LOGS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUN_EXPERIMENTS = True

# Key comparison first, followed by the two component controls.
CONFIGS = [
    dict(name="a5_layernorm_raw_wd1", residual_type="layernorm", head_type="raw", tau=None),
    # Reuse legacy sphere_cosine files: for a spherical residual stream,
    # the old cosine and paper-aligned bounded formulas are identical.
    dict(name="a5_sphere_bounded_wd1", storage_name="a5_sphere_cosine_wd1", residual_type="spherical", head_type="bounded", tau=10.0),
    dict(name="a5_sphere_raw_wd1", residual_type="spherical", head_type="raw", tau=None),
    # New name prevents the completed strict-cosine LayerNorm run from being reused.
    dict(name="a5_layernorm_bounded_wd1", residual_type="layernorm", head_type="bounded", tau=10.0),
]

for cfg in CONFIGS:
    cfg.update(
        d_model=D_MODEL,
        n_heads=NUM_HEADS,
        mlp_dim=MLP_DIM,
        n_layers=N_LAYERS,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        betas=ADAM_BETAS,
        max_epochs=MAX_EPOCHS,
        seeds=list(SEEDS),
    )

print("device:", DEVICE)
print("group:", GROUP_NAME, "size:", GROUP_SIZE)
print("ordered pairs:", GROUP_SIZE ** 2)
print("train pairs:", int(FRAC_TRAIN * GROUP_SIZE ** 2))


# ============================================================
# REPRODUCIBILITY AND A5 COMPOSITION TABLE
# ============================================================

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def permutation_parity(perm):
    inversions = sum(
        perm[i] > perm[j]
        for i in range(len(perm))
        for j in range(i + 1, len(perm))
    )
    return inversions % 2


def build_a5():
    elements = np.asarray(
        [perm for perm in itertools.permutations(range(DEGREE)) if permutation_parity(perm) == 0],
        dtype=np.int16,
    )
    assert len(elements) == GROUP_SIZE

    powers = DEGREE ** np.arange(DEGREE, dtype=np.int64)
    codes = (elements.astype(np.int64) * powers).sum(axis=1)
    code_to_id = np.full(DEGREE ** DEGREE, -1, dtype=np.int16)
    code_to_id[codes] = np.arange(GROUP_SIZE, dtype=np.int16)

    composition = np.empty((GROUP_SIZE, GROUP_SIZE), dtype=np.int16)
    for left_id, left in enumerate(elements):
        # Convention: (left o right)[i] = left[right[i]].
        outputs = left[elements]
        output_codes = (outputs.astype(np.int64) * powers).sum(axis=1)
        composition[left_id] = code_to_id[output_codes]

    return elements, composition


ELEMENTS, COMPOSITION = build_a5()
IDENTITY_ID = int(np.flatnonzero((ELEMENTS == np.arange(DEGREE)).all(axis=1))[0])

assert COMPOSITION.shape == (GROUP_SIZE, GROUP_SIZE)
assert COMPOSITION.min() >= 0
assert np.array_equal(COMPOSITION[IDENTITY_ID], np.arange(GROUP_SIZE))
assert np.array_equal(COMPOSITION[:, IDENTITY_ID], np.arange(GROUP_SIZE))

NONCOMMUTING_FRACTION = float(np.mean(COMPOSITION != COMPOSITION.T))
print("noncommuting fraction:", f"{NONCOMMUTING_FRACTION:.4f}")


def make_dataset(seed):
    set_seed(seed)
    rng = np.random.default_rng(seed)
    total_pairs = GROUP_SIZE ** 2
    train_size = int(FRAC_TRAIN * total_pairs)

    pair_ids = rng.permutation(total_pairs)
    train_ids = pair_ids[:train_size]
    test_ids = pair_ids[train_size:]

    def decode(ids):
        left = ids // GROUP_SIZE
        right = ids % GROUP_SIZE
        x = np.stack([left, right, np.full_like(left, OP_TOKEN)], axis=1)
        y = COMPOSITION[left, right].astype(np.int64)
        return torch.from_numpy(x).long(), torch.from_numpy(y).long()

    train_x, train_y = decode(train_ids)
    test_x, test_y = decode(test_ids)

    assert len(train_x) == train_size
    assert len(test_x) == total_pairs - train_size
    assert set(train_ids.tolist()).isdisjoint(test_ids.tolist())
    return train_x, train_y, test_x, test_y


In [ ]:
# ============================================================
# MODEL
# ============================================================

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_dim, residual_type):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.residual_type = residual_type

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)

        if residual_type == "layernorm":
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)

    def post_residual_norm(self, h, index):
        if self.residual_type == "spherical":
            return F.normalize(h, dim=-1)
        return self.norm1(h) if index == 1 else self.norm2(h)

    def forward(self, h):
        batch_size, seq_len, _ = h.shape
        q = self.W_Q(h).view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        k = self.W_K(h).view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        v = self.W_V(h).view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attention = scores.softmax(dim=-1)
        joined = (
            (attention @ v)
            .transpose(1, 2)
            .contiguous()
            .view(batch_size, seq_len, self.d_model)
        )

        h = self.post_residual_norm(h + self.W_O(joined), 1)
        h = self.post_residual_norm(
            h + self.mlp_out(F.relu(self.mlp_in(h))),
            2,
        )
        return h


class A5Transformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = deepcopy(cfg)
        self.residual_type = cfg["residual_type"]
        self.head_type = cfg["head_type"]
        self.tau = cfg["tau"]

        d_model = cfg["d_model"]
        self.token_embedding = nn.Embedding(GROUP_SIZE + 1, d_model)
        self.position_embedding = nn.Embedding(3, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=cfg["n_heads"],
                mlp_dim=cfg["mlp_dim"],
                residual_type=cfg["residual_type"],
            )
            for _ in range(cfg["n_layers"])
        ])
        self.unembed = nn.Linear(d_model, GROUP_SIZE, bias=False)

        if self.residual_type == "layernorm":
            self.input_norm = nn.LayerNorm(d_model)

    def input_norm_fn(self, h):
        if self.residual_type == "spherical":
            return F.normalize(h, dim=-1)
        return self.input_norm(h)

    def head_logits(self, h_final):
        if self.head_type == "bounded":
            # Paper-aligned head: normalize unembedding rows only.
            # Spherical h_final is already unit norm; LayerNorm h_final is not.
            weights = F.normalize(self.unembed.weight, dim=1)
            return F.linear(h_final, weights) * float(self.tau)
        if self.head_type == "raw":
            return self.unembed(h_final)
        raise ValueError(f"Unknown head_type: {self.head_type}")

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        h = self.token_embedding(x) + self.position_embedding(positions)
        h = self.input_norm_fn(h)

        for block in self.blocks:
            h = block(h)

        return self.head_logits(h[:, 2])


# Static smoke checks for all four operational combinations.
smoke_x, _, _, _ = make_dataset(seed=1)
for cfg in CONFIGS:
    set_seed(1)
    model = A5Transformer(cfg).cpu()
    with torch.no_grad():
        logits = model(smoke_x[:8])
    assert logits.shape == (8, GROUP_SIZE)
    assert torch.isfinite(logits).all()
    if cfg["head_type"] == "bounded":
        with torch.no_grad():
            row_norms = model.unembed.weight.norm(dim=1)
        assert torch.isfinite(row_norms).all()
        normalized_rows = F.normalize(model.unembed.weight, dim=1).norm(dim=1)
        assert torch.allclose(normalized_rows, torch.ones_like(normalized_rows), atol=1e-5)
        if cfg["residual_type"] == "spherical":
            with torch.no_grad():
                bounded_logits = model(smoke_x[:8])
            assert bounded_logits.abs().max().item() <= float(cfg["tau"]) + 1e-5
    print(cfg["name"], "parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
# ============================================================
# TRAINING, EVALUATION, AND RESUME
# ============================================================

def atomic_torch_save(obj, path):
    tmp_path = path + ".tmp"
    torch.save(obj, tmp_path)
    os.replace(tmp_path, path)


def atomic_json_dump(obj, path):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as handle:
        json.dump(obj, handle, indent=2)
    os.replace(tmp_path, path)


@torch.no_grad()
def evaluate(model, x, y):
    model.eval()
    x_device = x.to(DEVICE)
    y_device = y.to(DEVICE)
    logits = model(x_device)
    loss = F.cross_entropy(logits, y_device).item()
    accuracy = logits.argmax(dim=-1).eq(y_device).float().mean().item()
    return loss, accuracy


@torch.no_grad()
def evaluate_swap(model, test_x, test_y):
    model.eval()
    swapped_x = test_x.clone()
    swapped_x[:, 0] = test_x[:, 1]
    swapped_x[:, 1] = test_x[:, 0]

    left = test_x[:, 0].numpy()
    right = test_x[:, 1].numpy()
    swapped_y = torch.from_numpy(COMPOSITION[right, left].astype(np.int64)).long()

    original_pred = model(test_x.to(DEVICE)).argmax(-1).cpu()
    swapped_pred = model(swapped_x.to(DEVICE)).argmax(-1).cpu()
    noncommuting = test_y.ne(swapped_y)

    def masked_mean(values, mask):
        return values[mask].float().mean().item() if mask.any() else None

    return {
        "swap_original_accuracy": original_pred.eq(test_y).float().mean().item(),
        "swap_swapped_accuracy": swapped_pred.eq(swapped_y).float().mean().item(),
        "swap_prediction_invariance": original_pred.eq(swapped_pred).float().mean().item(),
        "noncommuting_fraction": noncommuting.float().mean().item(),
        "noncommuting_original_accuracy": masked_mean(original_pred.eq(test_y), noncommuting),
        "noncommuting_swapped_accuracy": masked_mean(swapped_pred.eq(swapped_y), noncommuting),
        "noncommuting_prediction_invariance": masked_mean(
            original_pred.eq(swapped_pred), noncommuting
        ),
    }


def first_onset(history, key, threshold):
    for epoch, value in zip(history["epoch"], history[key]):
        if value >= threshold:
            return epoch
    return None


def run_signature(cfg, seed):
    return {
        "group": GROUP_NAME,
        "degree": DEGREE,
        "frac_train": FRAC_TRAIN,
        "seed": seed,
        "residual_type": cfg["residual_type"],
        "head_type": cfg["head_type"],
        "tau": cfg["tau"],
        "d_model": cfg["d_model"],
        "n_heads": cfg["n_heads"],
        "mlp_dim": cfg["mlp_dim"],
        "n_layers": cfg["n_layers"],
        "lr": cfg["lr"],
        "weight_decay": cfg["weight_decay"],
        "betas": list(cfg["betas"]),
    }


def run_one(cfg, seed):
    signature = run_signature(cfg, seed)
    storage_name = cfg.get("storage_name", cfg["name"])
    run_name = (
        f'{storage_name}_frac{int(100 * FRAC_TRAIN)}'
        f'_d{cfg["d_model"]}_m{cfg["mlp_dim"]}_L{cfg["n_layers"]}_seed{seed}'
    )
    run_dir = os.path.join(SAVE_ROOT, storage_name)
    os.makedirs(run_dir, exist_ok=True)

    final_json_path = os.path.join(run_dir, run_name + ".json")
    final_model_path = os.path.join(run_dir, run_name + ".pt")
    recovery_path = os.path.join(run_dir, run_name + "_recovery.pt")

    if os.path.exists(final_json_path) and os.path.exists(final_model_path):
        with open(final_json_path) as handle:
            saved = json.load(handle)
        summary = dict(saved["summary"])
        # Report legacy spherical cosine files under the paper-aligned label.
        # Their computation is identical because spherical h_final is unit norm.
        summary.update(signature)
        summary["config"] = cfg["name"]
        print("skip completed:", run_name, "->", cfg["name"])
        return summary

    train_x, train_y, test_x, test_y = make_dataset(seed)
    set_seed(seed)
    model = A5Transformer(cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        betas=cfg["betas"],
    )

    history = {
        "epoch": [],
        "train_acc": [],
        "test_acc": [],
        "train_loss": [],
        "test_loss": [],
    }
    start_epoch = 0
    stable_99 = 0
    test_99_epoch = None
    stop_reason = "max_epochs"

    if os.path.exists(recovery_path):
        state = torch.load(recovery_path, map_location=DEVICE)
        state_signature = state["signature"]
        signature_matches = state_signature == signature
        if cfg["residual_type"] == "spherical" and cfg["head_type"] == "bounded":
            legacy_signature = dict(signature)
            legacy_signature["head_type"] = "cosine"
            signature_matches = signature_matches or state_signature == legacy_signature
        if not signature_matches:
            raise ValueError("Recovery checkpoint signature does not match this run")
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        history = state["history"]
        stable_99 = state["stable_99"]
        test_99_epoch = state["test_99_epoch"]
        start_epoch = state["epoch"] + 1
        print("resume:", run_name, "from epoch", start_epoch)

    train_x_device = train_x.to(DEVICE)
    train_y_device = train_y.to(DEVICE)
    started = time.time()
    last_epoch = start_epoch - 1

    progress = tqdm(
        range(start_epoch, cfg["max_epochs"]),
        desc=run_name,
        leave=False,
    )
    for epoch in progress:
        last_epoch = epoch
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(train_x_device)
        loss = F.cross_entropy(logits, train_y_device)

        if not torch.isfinite(loss):
            stop_reason = "non_finite_loss"
            break

        loss.backward()
        optimizer.step()

        should_log = epoch % LOG_EVERY == 0 or epoch == cfg["max_epochs"] - 1
        if should_log:
            train_loss, train_acc = evaluate(model, train_x, train_y)
            test_loss, test_acc = evaluate(model, test_x, test_y)

            history["epoch"].append(epoch)
            history["train_acc"].append(train_acc)
            history["test_acc"].append(test_acc)
            history["train_loss"].append(train_loss)
            history["test_loss"].append(test_loss)
            progress.set_postfix(train=f"{train_acc:.3f}", test=f"{test_acc:.3f}")

            if test_99_epoch is None and test_acc >= 0.99:
                test_99_epoch = epoch
            stable_99 = stable_99 + 1 if test_acc >= 0.99 else 0

            if (
                stable_99 >= STABLE_99_LOGS
                and test_99_epoch is not None
                and epoch >= test_99_epoch + POST_99_GRACE
            ):
                stop_reason = "stable_test_99_post_grace"
                break

        if epoch > 0 and epoch % CHECKPOINT_EVERY == 0:
            atomic_torch_save(
                {
                    "epoch": epoch,
                    "signature": signature,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "history": history,
                    "stable_99": stable_99,
                    "test_99_epoch": test_99_epoch,
                },
                recovery_path,
            )

    elapsed_seconds = time.time() - started
    train_loss, train_acc = evaluate(model, train_x, train_y)
    test_loss, test_acc = evaluate(model, test_x, test_y)
    swap_metrics = evaluate_swap(model, test_x, test_y)

    summary = {
        **signature,
        "config": cfg["name"],
        "train_size": len(train_x),
        "test_size": len(test_x),
        "max_epochs": cfg["max_epochs"],
        "last_epoch": last_epoch,
        "stop_reason": stop_reason,
        "elapsed_seconds": elapsed_seconds,
        "final_train_loss": train_loss,
        "final_test_loss": test_loss,
        "final_train_acc": train_acc,
        "final_test_acc": test_acc,
        "peak_test_acc": max(history["test_acc"]) if history["test_acc"] else test_acc,
        **swap_metrics,
    }

    for label, threshold in [("95", 0.95), ("99", 0.99)]:
        train_epoch = first_onset(history, "train_acc", threshold)
        test_epoch = first_onset(history, "test_acc", threshold)
        summary[f"train_epoch_{label}"] = train_epoch
        summary[f"test_epoch_{label}"] = test_epoch
        summary[f"gap_{label}"] = (
            None if train_epoch is None or test_epoch is None else test_epoch - train_epoch
        )

    atomic_torch_save(
        {
            "model_state_dict": model.state_dict(),
            "config": cfg,
            "signature": signature,
            "summary": summary,
        },
        final_model_path,
    )
    atomic_json_dump(
        {
            "summary": summary,
            "history": history,
            "config": cfg,
            "signature": signature,
        },
        final_json_path,
    )
    if os.path.exists(recovery_path):
        os.remove(recovery_path)

    print(
        "saved:", run_name,
        "test99:", summary["test_epoch_99"],
        "peak:", f'{summary["peak_test_acc"]:.4f}',
    )
    return summary


In [ ]:
# ============================================================
# RUNNER AND REBUTTAL TABLES
# ============================================================

if not RUN_EXPERIMENTS:
    print("Experiments disabled (RUN_EXPERIMENTS=False).")
else:
    rows = []
    # Seed-first order yields a complete matched four-way result as early as possible.
    for seed in SEEDS:
        for cfg in CONFIGS:
            rows.append(run_one(cfg, seed))
            pd.DataFrame(rows).to_csv(
                os.path.join(SAVE_ROOT, "summary_incremental.csv"),
                index=False,
            )

    results = pd.DataFrame(rows).sort_values(["config", "seed"])
    results.to_csv(os.path.join(SAVE_ROOT, "summary_final.csv"), index=False)

    aggregate = (
        results.groupby(["residual_type", "head_type"], dropna=False)
        .agg(
            seeds=("seed", "count"),
            reached_95=("test_epoch_95", lambda values: int(values.notna().sum())),
            reached_99=("test_epoch_99", lambda values: int(values.notna().sum())),
            mean_test_95=("test_epoch_95", "mean"),
            std_test_95=("test_epoch_95", "std"),
            mean_test_99=("test_epoch_99", "mean"),
            std_test_99=("test_epoch_99", "std"),
            mean_gap_95=("gap_95", "mean"),
            mean_gap_99=("gap_99", "mean"),
            mean_peak_test_acc=("peak_test_acc", "mean"),
            mean_noncommuting_accuracy=("noncommuting_original_accuracy", "mean"),
            mean_swap_invariance=("noncommuting_prediction_invariance", "mean"),
            mean_elapsed_seconds=("elapsed_seconds", "mean"),
        )
        .reset_index()
    )
    aggregate.to_csv(
        os.path.join(SAVE_ROOT, "configuration_summary.csv"),
        index=False,
    )

    print("\nPer-run results")
    show_columns = [
        "config", "seed", "stop_reason", "train_epoch_99", "test_epoch_99",
        "gap_99", "peak_test_acc", "noncommuting_original_accuracy",
        "noncommuting_prediction_invariance", "elapsed_seconds",
    ]
    print(results[show_columns].to_string(index=False))

    print("\nConfiguration summary")
    print(aggregate.to_string(index=False))
